The codes used to make the figures in Vasenkari et al.: Estimation of Carbon Fluxes in Urban Vegetation Using Machine Learning Emulators of the Land Surface Model JSBACH

Model data used can be found here: https://doi.org/10.57707/fmi-b2share.881fb-d5p61

EC measurements are from The urban measurement station SMEAR III (järvi et al. 2009).

We used the Level 2 SIF data product TROPOSIF from the TROPOspheric Monitoring Instrument (TROPOMI) aboard the Copernicus Sentinel-5P mission (Guanter et al. 2021).

The fractional coverages used to calculate the Kumpula and Helsinki emulator and JSBACH flux values are based on a land cover dataset provided by the Helsinki (Region Environmental Services Authority https://www.hsy.fi/en/environmental-information/open-data/avoin-data---sivut/helsinki-region-land-cover-dataset/) and rasterized to a 50m grid.

References:
Järvi, L., Hannuniemi, H., Hussein, T., Junninen, H., Aalto, P., Hillamo, R., Mäkelä, T., Keronen,
P., Siivola, E., Vesala, T., Kulmala, M., 2009. The urban measurement station smear iii:
continuous monitoring of air pollution and surface-atmosphere interactions in helsinki, finland.
Boreal Environment Research 14 (2009) Suppl. A, 86–109.

Guanter, L., Bacour, C., Schneider, A., Aben, I., van Kempen, T.A., Maignan, F., Retscher, C.,
Köhler, P., Frankenberg, C., Joiner, J., et al., 2021. The troposif global sun-induced fluorescence
dataset from the sentinel-5p tropomi mission. Earth System Science Data 13, 5423–5440.



In [ ]:
#Figure 1, Vasenkari et al.
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from scipy import stats
import matplotlib.ticker as ticker
import calendar
import locale
from collections import OrderedDict

DATA_PATH = "../path/to/data"

jsbach_file = DATA_PATH + "Vasenkari_etal_2026_JSBACH_NEE_GPP.csv"
daily_file = DATA_PATH + "Vasenkari_etal_2026_daily_emulator_NEE_GPP.csv"
monthly_file = DATA_PATH + "Vasenkari_etal_2026_monthly_emulator_NEE_GPP.csv"

jsbach = pd.read_csv(jsbach_file, parse_dates=["date"])
daily = pd.read_csv(daily_file, parse_dates=["date"])
monthly = pd.read_csv(monthly_file, parse_dates=["date"])

co2_to_C = 12.01 / 44.01  # gC per gCO2

for df in [daily, monthly]:
    flux_cols = [c for c in df.columns if ("GPP_" in c or "NEE_" in c)]
    df[flux_cols] = df[flux_cols] * co2_to_C

jsbach_cols = [c for c in jsbach.columns if ("GPP_" in c or "NEE_" in c)]
jsbach[jsbach_cols] = jsbach[jsbach_cols] * co2_to_C

pfts = ["birch", "conif", "grass", "agri_grass", "cereal"]
pft_labels = ["Deciduous", "Coniferous", "Lawn", "Agricultural grass", "Cereal"]
col_titles = ["GPP daily", "NEE daily", "GPP monthly", "NEE monthly"]

years_to_plot = list(range(2016, 2025))

daily = daily[daily['date'].dt.year.isin(years_to_plot)].copy()
monthly = monthly[monthly['date'].dt.year.isin(years_to_plot)].copy()
jsbach = jsbach[jsbach['date'].dt.year.isin(years_to_plot)].copy()

font_metrics = 10
title_fontsize = 15
alpha = 0.8
dotsize = 11
boxalpha = 0.6
bigalpha = 0.8
bigdotsize = 25


month_colors = {
    1: "#F5B82A", 2: "#f4aeae", 3: "#f6eb94", 4: "#64ee92", 5: "#fe84c3",
    6: "#BD140E", 7: "#004084", 8: "#FF8E03", 9: "#9265C6", 10: "#73CBFA",
    11: "#770EBD", 12: "#524A35"
}

def get_season(month):
    if month in [12,1,2]: return "winter"
    elif month in [3,4,5]: return "spring"
    elif month in [6,7,8]: return "summer"
    else: return "autumn"

color_mode = "month"   

fig, axs = plt.subplots(len(pfts), 4, figsize=(12, 12), sharex=False, sharey=False)

for row, pft in enumerate(pfts):
    
    jsb_gpp_col = f"GPP_{pft}_JSBACH"
    jsb_nee_col = f"NEE_{pft}_JSBACH"

    if (jsb_gpp_col not in jsbach.columns) or (jsb_nee_col not in jsbach.columns):
        raise KeyError(f"Missing JSBACH columns for PFT '{pft}': expected {jsb_gpp_col}, {jsb_nee_col}")

    jsb_daily = jsbach[["date", jsb_gpp_col, jsb_nee_col]].rename(columns={jsb_gpp_col: "GPP", jsb_nee_col: "NEE"})
    jsb_monthly = jsb_daily.set_index("date").resample("MS").mean().reset_index()

    
    emu_gpp_daily_col = f"GPP_{pft}_daily_emulator"
    emu_nee_daily_col = f"NEE_{pft}_daily_emulator"
    emu_gpp_month_col = f"GPP_{pft}_monthly_emulator"
    emu_nee_month_col = f"NEE_{pft}_monthly_emulator"

    for colname in [emu_gpp_daily_col, emu_nee_daily_col, emu_gpp_month_col, emu_nee_month_col]:
        if colname not in (daily.columns.tolist() + monthly.columns.tolist()):
       
            pass

    emu_daily_gpp = daily[["date", emu_gpp_daily_col]].rename(columns={emu_gpp_daily_col: "emu_GPP"})
    emu_daily_nee = daily[["date", emu_nee_daily_col]].rename(columns={emu_nee_daily_col: "emu_NEE"})
    emu_monthly_gpp = monthly[["date", emu_gpp_month_col]].rename(columns={emu_gpp_month_col: "emu_GPP_m"})
    emu_monthly_nee = monthly[["date", emu_nee_month_col]].rename(columns={emu_nee_month_col: "emu_NEE_m"})

   
    for col, (jsb_df, emu_df, y_col) in enumerate([
        (jsb_daily, emu_daily_gpp, "GPP"),
        (jsb_daily, emu_daily_nee, "NEE"),
        (jsb_monthly, emu_monthly_gpp, "GPP"),
        (jsb_monthly, emu_monthly_nee, "NEE")
    ]):
        merged = pd.merge(jsb_df, emu_df, on="date", how="inner")

        
        merged["PFT"] = pft
        merged["PFT_label"] = pft_labels[row]

       
        if col < 2:
            emu_col = "emu_GPP" if y_col == "GPP" else "emu_NEE"
        else:
            emu_col = "emu_GPP_m" if y_col == "GPP" else "emu_NEE_m"

        
        if emu_col not in merged.columns or merged.empty:
            axs[row, col].text(0.5, 0.5, "No data", ha="center", va="center")
            axs[row, col].set_axis_off()
            continue

        
        if color_mode == "season":
            merged["group"] = merged["date"].dt.month.apply(get_season)
            color_dict = {"winter":"#5DADE2","spring":"#58D68D","summer":"#F5B041","autumn":"#AF7AC5"}
        else:
            merged["group"] = merged["date"].dt.month
            color_dict = month_colors

        
        for group, gdf in merged.groupby("group"):
            size = bigdotsize if col >= 2 else dotsize
            alpha_val = bigalpha if col >= 2 else alpha
            axs[row, col].scatter(
                gdf[y_col], gdf[emu_col],
                s=size, alpha=alpha_val,
                color=color_dict[group],
                label=group if row == 0 else None,
                edgecolors="none"
            )

       
        vmin = min(merged[y_col].min(), merged[emu_col].min())
        vmax = max(merged[y_col].max(), merged[emu_col].max())
        pad = (vmax - vmin) * 0.02 if vmax > vmin else 1.0
        axs[row, col].plot([vmin - pad, vmax + pad], [vmin - pad, vmax + pad], 'k--', lw=1)

       
        r2 = r2_score(merged[y_col], merged[emu_col])
        rmse = np.sqrt(mean_squared_error(merged[y_col], merged[emu_col]))
        bias = (merged[emu_col] - merged[y_col]).mean()

        axs[row, col].text(0.05, 0.95,
            f"R²={r2:.2f}\nRMSE={rmse:.2f}\nBias={bias:.2f}",
            transform=axs[row, col].transAxes,
            va='top', ha='left', fontsize=font_metrics,
            bbox=dict(facecolor='white', alpha=boxalpha, edgecolor='none')
        )

        if row == 0:
            axs[row, col].set_title(col_titles[col], fontsize=title_fontsize)

        axs[row, col].grid(True, linestyle='--', alpha=0.7)
        axs[row, col].set_axisbelow(True)

    axs[row, 0].set_ylabel(pft_labels[row], fontsize=title_fontsize)

fig.text(0.04, 0.5, "Emulator [gC m$^{-2}$ d$^{-1}$]", va='center', rotation='vertical', fontsize=15)
fig.text(0.5, 0.03, "JSBACH [gC m$^{-2}$ d$^{-1}$]", va='center', rotation='horizontal', fontsize=15)

plt.subplots_adjust(left=0.08, right=0.98, top=0.97, bottom=0.07, wspace=0.2, hspace=0.15)


handles, labels = axs[0,0].get_legend_handles_labels()
legend_dict = OrderedDict()
for h, l in zip(handles, labels):
    if l not in legend_dict:
        legend_dict[l] = h
handles = list(legend_dict.values())
labels = list(legend_dict.keys())


labels = [calendar.month_abbr[int(l)] for l in labels]
legend_title = "Month"


fig.legend(handles, labels, title=legend_title, loc="lower right",
           bbox_to_anchor=(1.07, 0.065), fontsize=10, title_fontsize=10, frameon=True, ncol=1, markerscale=2)


plt.show()

In [ ]:
#Figure 2, Vasenkari et al.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict

DATA_PATH = "../path/to/data"

jsbach_file = DATA_PATH + "Vasenkari_etal_2026_JSBACH_NEE_GPP.csv"
daily_file = DATA_PATH + "Vasenkari_etal_2026_daily_emulator_NEE_GPP.csv"
monthly_file = DATA_PATH + "Vasenkari_etal_2026_monthly_emulator_NEE_GPP.csv"

jsbach = pd.read_csv(jsbach_file, parse_dates=["date"])
daily = pd.read_csv(daily_file, parse_dates=["date"])
monthly = pd.read_csv(monthly_file, parse_dates=["date"])

co2_to_C = 12.01 / 44.01

for df in [jsbach, daily, monthly]:

    flux_cols = [c for c in df.columns if ("GPP_" in c or "NEE_" in c)]

    df[flux_cols] = df[flux_cols] * co2_to_C
    

pfts = ["birch","conif","grass","agri_grass","cereal"]
labels = ["Deciduous","Coniferous","Lawn","Agricultural\n grass","Cereal"]

years_all = list(range(2016,2025))
years_dry = [2019,2022,2023,2024]
years_control = [2016,2017,2018,2020,2021]

rename_jsbach={}
rename_daily={}
rename_monthly={}

for p in pfts:

    rename_jsbach[f"GPP_{p}_JSBACH"]=f"GPP_{p}"
    rename_jsbach[f"NEE_{p}_JSBACH"]=f"NEE_{p}"

    rename_daily[f"GPP_{p}_daily_emulator"]=f"GPP_{p}"
    rename_daily[f"NEE_{p}_daily_emulator"]=f"NEE_{p}"

    rename_monthly[f"GPP_{p}_monthly_emulator"]=f"GPP_{p}"
    rename_monthly[f"NEE_{p}_monthly_emulator"]=f"NEE_{p}"

jsbach = jsbach.rename(columns=rename_jsbach)
daily = daily.rename(columns=rename_daily)
monthly = monthly.rename(columns=rename_monthly)

def compute_means(variable, years):

    js = jsbach[jsbach.date.dt.year.isin(years)]
    d = daily[daily.date.dt.year.isin(years)]
    m = monthly[monthly.date.dt.year.isin(years)].copy()

    m["days"]=m.date.dt.days_in_month

    js_mean,js_std=[],[]
    d_mean,d_std=[],[]
    m_mean,m_std=[],[]

    for p in pfts:

        # JSBACH yearly sums
        y_js = js.groupby(js.date.dt.year)[f"{variable}_{p}"].sum()
        js_mean.append(y_js.mean())
        js_std.append(y_js.std())

        # Daily emulator
        y_d = d.groupby(d.date.dt.year)[f"{variable}_{p}"].sum()
        d_mean.append(y_d.mean())
        d_std.append(y_d.std())

        # Monthly emulator → convert to totals
        m[f"{variable}_{p}_tot"] = m[f"{variable}_{p}"]*m["days"]
        y_m = m.groupby(m.date.dt.year)[f"{variable}_{p}_tot"].sum()

        m_mean.append(y_m.mean())
        m_std.append(y_m.std())

    return (js_mean,js_std,d_mean,d_std,m_mean,m_std)

gpp_all = compute_means("GPP",years_all)
nee_all = compute_means("NEE",years_all)

gpp_dry = compute_means("GPP",years_dry)
nee_dry = compute_means("NEE",years_dry)

gpp_ctrl = compute_means("GPP",years_control)
nee_ctrl = compute_means("NEE",years_control)

fig,axs = plt.subplots(3,2,figsize=(11,11),sharex=True)

x=np.arange(len(pfts))
w=0.25

colors=["#882828","#2694C3","#483C32"]

def plot_row(ax,data,title=None):

    js_mean,js_std,d_mean,d_std,m_mean,m_std=data

    ax.bar(x-w,js_mean,w,yerr=js_std,label="JSBACH",color=colors[0],capsize=5,zorder=2)
    ax.bar(x,d_mean,w,yerr=d_std,label="Emulator daily",color=colors[1],capsize=5,zorder=2)
    ax.bar(x+w,m_mean,w,yerr=m_std,label="Emulator monthly",color=colors[2],capsize=5,zorder=2)

    ax.grid(axis="y",alpha=0.5,zorder=0)

    if title:
        ax.set_title(title,fontsize=15)

# rows
plot_row(axs[0,0],gpp_all,"GPP")
plot_row(axs[0,1],nee_all,"NEE")

plot_row(axs[1,0],gpp_dry)
plot_row(axs[1,1],nee_dry)

plot_row(axs[2,0],gpp_ctrl)
plot_row(axs[2,1],nee_ctrl)

# axis formatting
axs[0,0].set_ylabel("All years")
axs[1,0].set_ylabel("Dry years")
axs[2,0].set_ylabel("Control years")

for ax in axs[:,1]:
    ax.axhline(0,color="black",linewidth=1.5)

# hide x labels except bottom row
for ax in axs[:-1, :].flatten():
    ax.tick_params(labelbottom=False)

axs[2,0].set_xticks(x)
axs[2,1].set_xticks(x)

axs[2,0].set_xticklabels(labels,rotation=25)
axs[2,1].set_xticklabels(labels,rotation=25)

axs[:,1][0].set_ylim(-200,195)
axs[:,1][1].set_ylim(-200,195)
axs[:,1][2].set_ylim(-200,195)

axs[:,1][0].set_ylim(-200,195)
axs[:,1][1].set_ylim(-200,195)
axs[:,1][2].set_ylim(-200,195)


# shared labels
fig.text(-0.02,0.5,"Average annual sum [gC m$^{-2}$ y$^{-1}$]",va="center",rotation="vertical",fontsize=15)
fig.text(0.5,0.03,"PFT",va="center",fontsize=15)

# shared legend
handles,labels=axs[0,0].get_legend_handles_labels()
fig.legend(handles,labels,title="Model",loc="upper center",bbox_to_anchor=(0.5,1.05),ncol=3)

plt.tight_layout()


plt.show()

In [ ]:
#Figure 3, Vasenkari et al.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date
import math

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date
import math

from matplotlib.ticker import FuncFormatter

co2_to_C = 12.01 / 44.01  # Convert gCO2 to gC


df_kumpula_jsbach=pd.read_csv("-----------your Kumpula JSBACH path-------------")
df_kumpula_jsbach["Date"]=pd.to_datetime(df_kumpula_jsbach["Date"]) 
df_kumpula_jsbach["Average[gCO₂ m⁻² d⁻¹]"]=df_kumpula_jsbach["Average[gCO₂ m⁻² d⁻¹]"].multiply(co2_to_C)


df_kumpula_emulator=pd.read_csv("-----------your Kumpula emulator path-------------")
df_kumpula_emulator["Date"]=pd.to_datetime(df_kumpula_emulator["Date"])
df_kumpula_emulator["Average[gCO₂ m⁻² d⁻¹]"]=df_kumpula_emulator["Average[gCO₂ m⁻² d⁻¹]"].multiply(co2_to_C)

df_EC=pd.read_csv("-----------your EC path-------------")

# Hard-coded English month names (independent of locale)
english_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def english_month(x, pos):
    """Formatter: return English month abbreviation based on month number"""
    month_num = mdates.num2date(x).month  # integer 1–12
    return english_months[month_num - 1]

ncols=4
fontsize=20
years = range(2016, 2024)  # 17 years
n_years = len(years)
nrows = math.ceil(n_years / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows+2), sharey=True, sharex=False)
axes = axes.flatten()

for i, year in enumerate(years):
    ax = axes[i]

    # JSBACH
    ax.plot(df_kumpula_jsbach["Date"], df_kumpula_jsbach['Average[gCO₂ m⁻² d⁻¹]'],
            color="#34B8FF", linewidth=2.5, label="JSBACH daily", alpha=1.0, zorder=2)

    # Emulator
    ax.plot(df_kumpula_emulator["Date"], df_kumpula_emulator['Average[gCO₂ m⁻² d⁻¹]'],
            color="#BD140E", linewidth=2.5, label="Emulator daily", alpha=0.9, zorder=2)

    # Eddy covariance obs
    ax.plot(df_EC["Date"], df_EC["Daily sum (g C m-2 day-1)"],
            color="#242424", linestyle="", marker="x", markersize=7, mew=2, 
            label="EC obs.", zorder=3)


    # Year limits
    ax.set_xlim([date(year, 4, 27), date(year, 9, 30)])
    ax.set_ylim(-5.5, 4.75)


    # Grid lines
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(''))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonthday=2))
    ax.grid(True, which="major", linestyle="--", alpha=0.6)

    # Month labels only for bottom row
    row = i // ncols
    if row == nrows - 1:
        ax.xaxis.set_minor_formatter(FuncFormatter(english_month))
        ax.tick_params(axis="x", which="minor", labelsize=fontsize-2, rotation=45, length=0)
    else:
        ax.xaxis.set_minor_formatter(mdates.DateFormatter(''))
        ax.tick_params(axis="x", which="minor", length=0)

    ax.tick_params(axis="both", labelsize=fontsize-2)
    ax.set_title(str(year), fontsize=fontsize)


# Remove unused subplots
for j in range(n_years, len(axes)):
    fig.delaxes(axes[j])

# Common labels
fig.text(0.5, 0.04, "Month", ha="center", fontsize=fontsize+2)
fig.text(-0.01, 0.5, "NEE [gC m⁻² d⁻¹]", va="center", rotation="vertical", fontsize=fontsize+2)

# Legend once
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=fontsize)

plt.tight_layout(rect=[0, 0.07, 1.0, 0.92])


plt.show()





In [ ]:
#Figure 4, Vasenkari et al.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date
import math

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date
import math

from matplotlib.ticker import FuncFormatter

co2_to_C = 12.01 / 44.01  # Convert gCO2 to gC


df_helsinki_jsbach=pd.read_csv("-----------your Helsinki JSBACH path-------------")
df_helsinki_jsbach["Date"]=pd.to_datetime(df_helsinki_jsbach["Date"]) 
df_helsinki_jsbach["Average[gCO₂ m⁻² d⁻¹]"]=df_helsinki_jsbach["Average[gCO₂ m⁻² d⁻¹]"].multiply(co2_to_C)


df_helsinki_emulator=pd.read_csv("-----------your Helsinki emulator path-------------")
df_helsinki_emulator["Date"]=pd.to_datetime(df_helsinki_emulator["Date"])
df_helsinki_emulator["Average[gCO₂ m⁻² d⁻¹]"]=df_helsinki_emulator["Average[gCO₂ m⁻² d⁻¹]"].multiply(co2_to_C)

df_SIF=pd.read_csv("-----------your SIF path-------------")

# Hard-coded English month names (independent of locale)
english_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def english_month(x, pos):
    """Formatter: return English month abbreviation based on month number"""
    month_num = mdates.num2date(x).month  # integer 1–12
    return english_months[month_num - 1]

ncols=4
fontsize=20
years = range(2016, 2024)  # 17 years
n_years = len(years)
nrows = math.ceil(n_years / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows+2), sharey=True, sharex=False)
axes = axes.flatten()

for i, year in enumerate(years):
    ax = axes[i]

    # JSBACH
    ax.plot(df_helsinki_jsbach["Date"], df_helsinki_jsbach['Average[gCO₂ m⁻² d⁻¹]'],
            color="#34B8FF", linewidth=2.5, label="JSBACH daily", alpha=1.0, zorder=2)

    # Emulator
    ax.plot(df_helsinki_emulator["Date"], df_helsinki_emulator['Average[gCO₂ m⁻² d⁻¹]'],
            color="#BD140E", linewidth=2.5, label="Emulator daily", alpha=0.9, zorder=2)

    # SIF obs
    ax.plot(df_SIF["Date"], df_SIF["Daily sum (g C m-2 day-1)"],
            color="#242424", linestyle="", marker="x", markersize=7, mew=2, 
            label="EC obs.", zorder=3)


    # Year limits
    ax.set_xlim([date(year, 4, 27), date(year, 9, 30)])
    ax.set_ylim(-5.5, 4.75)


    # Grid lines
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(''))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonthday=2))
    ax.grid(True, which="major", linestyle="--", alpha=0.6)

    # Month labels only for bottom row
    row = i // ncols
    if row == nrows - 1:
        ax.xaxis.set_minor_formatter(FuncFormatter(english_month))
        ax.tick_params(axis="x", which="minor", labelsize=fontsize-2, rotation=45, length=0)
    else:
        ax.xaxis.set_minor_formatter(mdates.DateFormatter(''))
        ax.tick_params(axis="x", which="minor", length=0)

    ax.tick_params(axis="both", labelsize=fontsize-2)
    ax.set_title(str(year), fontsize=fontsize)


# Remove unused subplots
for j in range(n_years, len(axes)):
    fig.delaxes(axes[j])

# Common labels
fig.text(0.5, 0.04, "Month", ha="center", fontsize=fontsize+2)
fig.text(-0.01, 0.5, "NEE [gC m⁻² d⁻¹]", va="center", rotation="vertical", fontsize=fontsize+2)

# Legend once
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=fontsize)

plt.tight_layout(rect=[0, 0.07, 1.0, 0.92])


plt.show()

